# Pretraining of Swin Transformer Encoder

Autoencoder pretraining for 2D confocal microscopy patches.

**Notebook structure:**
1. Config (all tunables in one place)
2. Imports & device setup
3. Data loading & sanity checks
4. Model definition
5. Overfit sanity check

## 1. Configuration

All parameters live here. Change these cells, nothing else.

In [ ]:
from types import SimpleNamespace

In [ ]:

cfg = SimpleNamespace(
    seed=42,
    output_root="./outputs",
    tag="overfit_test",    
)

In [ ]:
data_cfg = SimpleNamespace(
    data_root="../../data/patches_128",       
    exclude_patterns=["KONTROLA"],
    val_split=0.1,
    batch_size=32,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
model_cfg = SimpleNamespace(
    in_channels=3,
    spatial_dims=2,
    img_size=128,
    feature_size=48,
    patch_size=2,
    window_size=7,
    dropout_path_rate=0.0,
    use_checkpoint=False,
)

In [ ]:
import torch.nn as nn

overfit_cfg = SimpleNamespace(
    lr=5e-4,
    weight_decay=0.05,          # SimMIM uses 0.05 (Xie et al. 2022, Sec 4.1)
    grad_clip_norm=1.0,         # SimMIM uses 1.0 (Xie et al. 2022); guards AMP + Swin early steps
    max_steps=2000,
    log_every=100,
    # Indices of patches to overfit on (pick a few diverse ones)
    patch_indices=[3, 4, 50, 1000],
    # Masking
    mask_ratio=0.25,
    mask_block_size=16,
    # Loss function: swap this to experiment with different losses.
    # nn.MSELoss()      -- standard L2, smooth gradients, can blur reconstructions
    # nn.L1Loss()       -- sharper reconstructions, less sensitive to outliers
    # nn.SmoothL1Loss() -- hybrid: L2 near zero, L1 far from zero
    loss_fn=None,
)

## 2. Imports & Device

In [ ]:
import os, sys

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm
from monai.networks.nets.swin_unetr import SwinTransformer

sys.path.insert(0, os.path.abspath("../.."))
from data_utils.patch_dataset import PatchDataset

In [ ]:
# Prevent OpenBLAS from spawning too many threads (causes hangs on HPC)
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK",
             os.environ.get("PBS_NUM_PPN", 4)))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(max(1, n_cpus // 2)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = torch.Generator().manual_seed(cfg.seed)
torch.manual_seed(cfg.seed)

if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")

## 3. Data Loading

In [ ]:
dataset = PatchDataset(
    root=data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
print(f"Total patches: {len(dataset)}")

sample = dataset[0]
print(f"Sample shape: {sample.shape}, dtype: {sample.dtype}")
print(f"Value range:  [{sample.min():.4f}, {sample.max():.4f}]")
assert sample.ndim == 3, f"Expected 3D tensor (C,H,W), got {sample.ndim}D"
assert 0 <= sample.min() and sample.max() <= 1.0 + 1e-6, "Values outside [0, 1]"

In [ ]:
n_val = int(len(dataset) * data_cfg.val_split)
n_train = len(dataset) - n_val
train_dataset, val_dataset = random_split(
    dataset, [n_train, n_val], generator=generator
)
print(f"Train: {n_train}, Validation: {n_val}")

train_loader = DataLoader(
    train_dataset,
    batch_size=data_cfg.batch_size,
    shuffle=True,
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=data_cfg.batch_size,
    shuffle=False,
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
data_cfg.n_val = n_val
data_cfg.n_train = n_train

## 4. Model Definition

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.nets.swin_unetr import SwinTransformer


class SimMIMSwin(nn.Module):
    """SimMIM-style masked image modeling on a Swin encoder.

    Reference: Xie et al., "SimMIM: A Simple Framework for Masked Image
    Modeling", CVPR 2022, arXiv:2111.09886.
    Official code: https://github.com/microsoft/SimMIM/blob/main/models/simmim.py

    Pipeline:
        x (B,C,H,W)  --apply mask in pixel space-->  x_masked
        x_masked     --SwinTransformer-->            z (B, enc_ch, S, S)
        z            --1x1 Conv + PixelShuffle-->    x_recon (B,C,H,W)
        loss         --L1 on masked positions only-->scalar

    Channel handling: this class is RGB-agnostic. C is taken from args.in_channels
    and threaded everywhere. The 3 fluorescence channels are treated as 3
    independent intensity maps. No ImageNet mean/std is applied.

    Deviations from canonical SimMIM are marked with `# DEVIATION:` comments.
    """

    def __init__(self, args):
        super().__init__()

        self.in_chans   = args.in_channels
        self.img_size   = args.img_size
        # Total spatial downsampling from input to the deepest Swin stage.
        # patch_embed downsamples by patch_size, then each of the 4 BasicLayers
        # in MONAI's SwinTransformer applies PatchMerging (downsample=True
        # by default), each halving spatial dims.
        # Total: patch_size * 2^4 = 2 * 16 = 32.
        # Verified empirically: z shape is (B, 768, 4, 4) for 128x128 input.
        # We need this for the PixelShuffle decoder.
        self.encoder_stride = args.patch_size * (2 ** 4)  # = 32 for patch_size=2
        assert args.img_size % self.encoder_stride == 0, \
            f"img_size {args.img_size} must be divisible by encoder_stride {self.encoder_stride}"

        # ------------------------------------------------------------------
        # Encoder: MONAI SwinTransformer, used unmodified.
        # We pick the deepest-stage feature map (index 4) as the bottleneck.
        # Without a hierarchical encoder we would have no SwinUNETR decoder
        # to attach later for fine-tuning, which is the whole point of using
        # this backbone.
        # ------------------------------------------------------------------
        patch_size  = (args.patch_size,)  * args.spatial_dims
        window_size = (args.window_size,) * args.spatial_dims

        self.swinViT = SwinTransformer(
            in_chans=args.in_channels,           # threaded from args, not hardcoded to 3
            embed_dim=args.feature_size,
            window_size=window_size,
            patch_size=patch_size,
            depths=[2, 2, 2, 2],
            num_heads=[3, 6, 12, 24],
            mlp_ratio=4.0,                       # standard Swin/SimMIM
            qkv_bias=True,                       # standard Swin/SimMIM
            drop_rate=0.0,
            attn_drop_rate=0.0,
            drop_path_rate=args.dropout_path_rate,
            norm_layer=nn.LayerNorm,
            use_checkpoint=args.use_checkpoint,
            spatial_dims=args.spatial_dims,
        )

        # Channels at the deepest stage: feature_size * 2^4 (four PatchMerging stages).
        enc_ch = args.feature_size * (2 ** 4)

        # ------------------------------------------------------------------
        # Decoder: SimMIM official "linear" head = 1x1 Conv + PixelShuffle.
        # The 1x1 Conv expands channels by stride^2 so PixelShuffle can
        # rearrange them back into (in_chans, H, W) at full resolution.
        # SimMIM ablation Table 6 shows that heavier decoders (inverse Swin,
        # 2-layer MLP) match this 1-layer head. Heavier = more params, no gain.
        # ------------------------------------------------------------------
        self.decoder = nn.Sequential(
            nn.Conv2d(
                in_channels=enc_ch,
                out_channels=self.encoder_stride ** 2 * self.in_chans,
                kernel_size=1,
            ),
            nn.PixelShuffle(self.encoder_stride),
        )

        # ------------------------------------------------------------------
        # Mask token.
        # DEVIATION: canonical SimMIM injects a learnable embed_dim vector at
        # the patch-embed output (embedding space). We inject in pixel space
        # to keep MONAI's SwinTransformer untouched. A single learnable
        # per-channel scalar is broadcast over all masked pixels, matching
        # SimMIM in spirit (one vector replaces every masked patch).
        # ------------------------------------------------------------------
        self.mask_token = nn.Parameter(torch.zeros(1, self.in_chans, 1, 1))
        nn.init.trunc_normal_(self.mask_token, mean=0.0, std=0.02)  # SimMIM init std

    # ----------------------------------------------------------------------
    # Encoder accessors. Kept separate so you can grab features later for
    # clustering / linear probing without rerunning the decoder.
    # ----------------------------------------------------------------------
    def encode(self, x):
        """Returns the deepest Swin feature map, (B, enc_ch, S, S)."""
        # MONAI's SwinTransformer returns a list [x0, x1, x2, x3, x4]
        # corresponding to the patch_embed output and the four BasicLayer
        # outputs. x4 is the deepest. Required by .contiguous() because
        # the layer outputs may be non-contiguous after windowing.
        return self.swinViT(x.contiguous())[4]

    def encode_pooled(self, x):
        """(B, enc_ch) global descriptor, useful for collapse / clustering checks."""
        return self.encode(x).mean(dim=(2, 3))

    # ----------------------------------------------------------------------
    # Forward.
    # mask: (B, 1, H, W) with 1 = masked, 0 = visible. This convention
    # matches your existing helpers (random_block_mask, simmim_l1_loss).
    # ----------------------------------------------------------------------
    def forward(self, x, mask):
        # Replace masked pixels with the learnable token (broadcast over H,W).
        # This is the pixel-space SimMIM masking.
        x_masked = x * (1.0 - mask) + self.mask_token * mask
        z = self.encode(x_masked)
        x_recon = self.decoder(z)
        return x_recon

## Losses

In [ ]:
def simmim_l1_loss(pred, target, mask):
    """SimMIM reconstruction loss.

    L1 between prediction and target, summed over masked pixel-channels,
    divided by (number of masked pixels * channels) for a per-element mean.

    Reference: Xie et al. 2022, eq. (1).
    Official: github.com/microsoft/SimMIM/blob/main/models/simmim.py#L60-L67

    Args:
        pred:   (B, C, H, W) reconstruction in [0, 1]-ish range
        target: (B, C, H, W) original image in [0, 1]
        mask:   (B, 1, H, W) with 1 = masked, broadcast over C
    Returns:
        scalar tensor
    """
    # Per-element absolute error, then zero out visible positions.
    # mask is (B,1,H,W) and broadcasts over channel dim.
    loss = (pred - target).abs() * mask                              # (B, C, H, W)
    # Denominator is (#masked pixels) * C. The +1e-8 guards against
    # the degenerate case of an empty mask.
    denom = mask.sum() * pred.shape[1] + 1e-8
    return loss.sum() / denom

In [ ]:
# Adapted from: https://github.com/microsoft/SimMIM/blob/main/data/data_simmim_pt.py
# Also: https://github.com/open-mmlab/mmselfsup/blob/master/mmselfsup/datasets/pipelines/transforms.py
# Differences vs source, all flagged with `# DEVIATION:`:
#   - returns torch.float (B,1,H,W) at PIXEL resolution rather than np.int 2D at
#     token resolution, because our mask token is in pixel space (see model)
#   - vectorized over the batch instead of per-sample (caller passes a full batch)
def random_block_mask(img, block_size, mask_ratio):
    """SimMIM-style random block mask.

    Returns: (B, 1, H, W) float tensor on img.device, 1 = masked, 0 = visible.
    """
    B, _, H, W = img.shape
    # DEVIATION: SimMIM asserts divisibility; we keep the assert because silent
    # truncation would produce a mask smaller than the image and the loss
    # broadcast would be wrong in subtle ways.
    assert H % block_size == 0 and W % block_size == 0, \
        f"H={H}, W={W} must be divisible by block_size={block_size}"

    gh, gw   = H // block_size, W // block_size
    n_blocks = gh * gw
    # SimMIM uses np.ceil here. round vs ceil differs by at most 1 block,
    # but we follow the published recipe to keep results comparable.
    n_mask = int(math.ceil(n_blocks * mask_ratio))

    # Vectorized random permutation per batch element via argsort of uniform noise.
    # Equivalent to running torch.randperm B times but avoids the Python loop.
    noise   = torch.rand(B, n_blocks, device=img.device)
    rank    = noise.argsort(dim=1)                       # (B, n_blocks)
    flat    = (rank < n_mask).float()                    # (B, n_blocks), 1 where masked
    mask    = flat.view(B, 1, gh, gw)
    mask    = F.interpolate(mask, scale_factor=block_size, mode="nearest")
    return mask                                          # (B, 1, H, W)

## Overfit

In [ ]:
# Build the overfit mini-batch from selected patch indices
x_fixed = torch.stack(
    [dataset[i] for i in overfit_cfg.patch_indices]
).to(device)
print(f"Overfit tensor: {x_fixed.shape}")

fig, axes = plt.subplots(1, len(overfit_cfg.patch_indices), figsize=(4 * len(overfit_cfg.patch_indices), 4))
for i, ax in enumerate(axes):
    img = x_fixed[i].cpu()
    ax.imshow(img.permute(1, 2, 0).numpy() if img.shape[0] >= 3 else img[0].numpy(), cmap="gray")
    ax.set_title(f"Patch idx={overfit_cfg.patch_indices[i]}")
    ax.axis("off")
plt.suptitle("Overfit set")
plt.tight_layout()
plt.show()

###  Overfit single mask frozen

In [ ]:
# Build, move to device, count params, verify shapes end-to-end on x_fixed.
model = SimMIMSwin(model_cfg).to(device)
overfit_cfg.loss_fun=simmim_l1_loss

n_total   = sum(p.numel() for p in model.parameters())
n_encoder = sum(p.numel() for p in model.swinViT.parameters())
n_decoder = sum(p.numel() for p in model.decoder.parameters())
print(f"Total params:   {n_total:,}")
print(f"Encoder params: {n_encoder:,}")
print(f"Decoder params: {n_decoder:,}  (1x1 Conv + PixelShuffle)")
print(f"Mask token:     {tuple(model.mask_token.shape)}")
print(f"Encoder stride: {model.encoder_stride}")

# Forward pass with a dummy mask just to confirm shapes line up.
with torch.no_grad():
    dummy_mask = torch.zeros(x_fixed.shape[0], 1, model_cfg.img_size, model_cfg.img_size, device=device)
    z = model.encode(x_fixed)
    recon = model(x_fixed, dummy_mask)
print(f"x_fixed:  {tuple(x_fixed.shape)}    in [{x_fixed.min():.3f}, {x_fixed.max():.3f}]")
print(f"z (deep): {tuple(z.shape)}     expected (B, {model_cfg.feature_size * 16}, "
      f"{model_cfg.img_size // model.encoder_stride}, {model_cfg.img_size // model.encoder_stride})")
print(f"recon:    {tuple(recon.shape)}    in [{recon.min():.3f}, {recon.max():.3f}]")
assert recon.shape == x_fixed.shape, "decoder did not return to input resolution"

In [ ]:
# Single mask, frozen for the whole run. If the model cannot drive loss to
# near zero on this configuration, something is wrong with the architecture
# or the optimizer, not the data.

torch.manual_seed(cfg.seed)
fixed_mask = random_block_mask(x_fixed, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio).to(device)
print(f"Fixed mask shape: {tuple(fixed_mask.shape)}, "
      f"masked fraction: {fixed_mask.mean().item():.3f} "
      f"(target {overfit_cfg.mask_ratio})")

In [ ]:
model = SimMIMSwin(model_cfg).to(device)  # fresh init for a clean sanity check
opt_test = torch.optim.AdamW(model.parameters(), lr=overfit_cfg.lr,
                             weight_decay=overfit_cfg.weight_decay,
                             betas=(0.9, 0.999))   # SimMIM uses AdamW (0.9, 0.999)
scaler_test = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

In [ ]:
max_steps = 2000  
loss_history = []

In [ ]:
model.train()
for step in tqdm(range(max_steps), desc="SimMIM overfit | fixed mask"):
    opt_test.zero_grad(set_to_none=True)
    with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
        recon = model(x_fixed, fixed_mask)
        loss = overfit_cfg.loss_fun(recon, x_fixed, fixed_mask)
    scaler_test.scale(loss).backward()
    # Gradient clipping is optional in SimMIM (paper uses 5.0). Kept here
    # because AMP + Swin can occasionally produce a large step early on.
    scaler_test.unscale_(opt_test)
    torch.nn.utils.clip_grad_norm_(model.parameters(), overfit_cfg.grad_clip_norm)
    scaler_test.step(opt_test)
    scaler_test.update()
    loss_history.append(loss.item())
    if step % overfit_cfg.log_every == 0:
        tqdm.write(f"step {step:4d} | loss {loss.item():.6f}")

print(f"final loss (fixed mask): {loss_history[-1]:.6f}")

In [ ]:
import copy
model_frozen = copy.deepcopy(model)
plot_loss_history = loss_history

### Overfit regenerated mask

In [ ]:
# Same 4 patches, but the mask is regenerated every step. The model must
# learn to reconstruct the patches under arbitrary masking, not memorize
# one (image, mask) pair. Loss will be higher than the fixed-mask run.

model = SimMIMSwin(model_cfg).to(device)  # fresh init again
opt_test = torch.optim.AdamW(model.parameters(), lr=overfit_cfg.lr,
                             weight_decay=overfit_cfg.weight_decay,
                             betas=(0.9, 0.999))
scaler_test = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

In [ ]:
max_steps = 2000
loss_history_refresh = []

In [ ]:
model.train()
for step in tqdm(range(max_steps), desc="SimMIM overfit | refresh mask"):
    mask = random_block_mask(x_fixed, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio).to(device)
    opt_test.zero_grad(set_to_none=True)
    with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
        recon = model(x_fixed, mask)
        loss = simmim_l1_loss(recon, x_fixed, mask)
    scaler_test.scale(loss).backward()
    scaler_test.unscale_(opt_test)
    torch.nn.utils.clip_grad_norm_(model.parameters(), overfit_cfg.grad_clip_norm)
    scaler_test.step(opt_test)
    scaler_test.update()
    loss_history_refresh.append(loss.item())
    if step % overfit_cfg.log_every == 0:
        tqdm.write(f"step {step:4d} | loss {loss.item():.6f}")

print(f"final loss (refresh mask): {loss_history_refresh[-1]:.6f}")

In [ ]:
plot_loss_history = loss_history_refresh

## Simple Validation of overfit

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(plot_loss_history)
plt.xlabel("Step")
plt.ylabel(f"Loss ({type(overfit_cfg.loss_fun).__name__})")
plt.title("Overfit loss curve")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visual reconstruction of the overfit set.
#
# Layout (rows × cols = 4 × n_patches):
#   row 0: original
#   row 1: masked input  (visible pixels only; masked regions blacked out)
#   row 2: reconstruction (clipped to [0,1] FOR DISPLAY ONLY — see comment below)
#   row 3: mask           (white = masked, black = visible)
#
# IMPORTANT: imshow renders 3-channel arrays as RGB (ch0=R, ch1=G, ch2=B).
# This is a false-color composite. Your channels are 3 fluorescence channels,
# not RGB. The model treats them as 3 independent intensity maps. If you want
# to inspect them as such, use the per-channel diagnostic cell below.

model.eval()
# Run inference in float32 (no autocast) so reconstruction values are stable
# for visualization. Training uses autocast; visualization should not.
with torch.no_grad():
    recon = model(x_fixed, fixed_mask)

n_patches = x_fixed.shape[0]
fig, axes = plt.subplots(4, n_patches, figsize=(4 * n_patches, 16))

for i in range(n_patches):
    orig   = x_fixed[i].cpu().permute(1, 2, 0).numpy()
    # Masked-input visualization: zero out masked pixels. Note the encoder
    # actually sees mask_token (a learnable value) at those positions, not
    # zeros — but zero is the cleanest visual indicator of "what was hidden".
    masked = (x_fixed[i] * (1.0 - fixed_mask[i])).cpu().permute(1, 2, 0).numpy()
    # Clip ONLY for display. The training loss sees the raw, unclipped recon,
    # which can drift slightly below 0 or above 1 because the SimMIM head has
    # no sigmoid. If recon.min()/max() are far outside [0,1], that itself is
    # diagnostic and is printed below.
    rec    = recon[i].clamp(0.0, 1.0).cpu().permute(1, 2, 0).numpy()
    mk     = fixed_mask[i, 0].cpu().numpy()  # (H, W), 1=masked

    axes[0, i].imshow(orig);              axes[0, i].set_title(f"patch {i}: original")
    axes[1, i].imshow(masked);            axes[1, i].set_title("masked input")
    axes[2, i].imshow(rec);               axes[2, i].set_title("reconstruction")
    axes[3, i].imshow(mk, cmap="gray", vmin=0, vmax=1)
    axes[3, i].set_title(f"mask ({mk.mean():.2f})")
    for r in range(4):
        axes[r, i].axis("off")

# Diagnostic: out-of-range reconstruction values mean the head is not yet
# saturated to the data range. Not an error, just useful to see.
print(f"recon range (raw, before clip): [{recon.min().item():.3f}, {recon.max().item():.3f}]")
print(f"orig range:                     [{x_fixed.min().item():.3f}, {x_fixed.max().item():.3f}]")
print(f"masked fraction (avg):          {fixed_mask.mean().item():.3f}")

plt.suptitle("SimMIM overfit reconstructions (3-channel composite, RGB display)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell A — per-sample loss + similarity diagnostics
# Uses fixed_mask so the loss values are directly comparable to the training
# curve. Reports both L1 (matches training loss) and MSE (kept from your
# original; useful as a second perspective sensitive to large outliers).
model.eval()
with torch.no_grad():
    recon = model(x_fixed, fixed_mask)
    C = x_fixed.shape[1]

    print("=== Per-sample masked L1 (matches training loss) ===")
    for i in range(len(x_fixed)):
        m = fixed_mask[i:i+1]
        l1 = ((recon[i:i+1] - x_fixed[i:i+1]).abs() * m).sum() / (m.sum() * C + 1e-8)
        print(f"  patch {i}: masked L1  = {l1.item():.6f}")

    print("\n=== Per-sample masked MSE (diagnostic only) ===")
    for i in range(len(x_fixed)):
        m = fixed_mask[i:i+1]
        mse = (((recon[i:i+1] - x_fixed[i:i+1]) ** 2) * m).sum() / (m.sum() * C + 1e-8)
        print(f"  patch {i}: masked MSE = {mse.item():.6f}")

    # Cosine similarity. If recons are all near 1.0 mutual similarity but
    # originals are not, the encoder is collapsing toward a shared output.
    n = recon.shape[0]
    rec_flat = recon.reshape(n, -1)
    org_flat = x_fixed.reshape(n, -1)

    print("\n=== Cosine similarity: reconstructions ===")
    for i in range(n):
        for j in range(i + 1, n):
            sim = F.cosine_similarity(rec_flat[i:i+1], rec_flat[j:j+1]).item()
            print(f"  recon[{i}] vs recon[{j}]: {sim:.4f}")

    print("\n=== Cosine similarity: originals (reference) ===")
    for i in range(n):
        for j in range(i + 1, n):
            sim = F.cosine_similarity(org_flat[i:i+1], org_flat[j:j+1]).item()
            print(f"  orig[{i}]  vs orig[{j}]:  {sim:.4f}")

In [ ]:
# Cell B — collapse check: seen vs unseen patch
# Uses a ZERO mask so the comparison isolates the encoder->decoder mapping
# from any randomness in masking. With a random mask, a non-zero diff could
# come from the inputs OR from the masks differing, which is not what we
# want to measure.
model.eval()
with torch.no_grad():
    unseen_idx = 100
    unseen = dataset[unseen_idx].unsqueeze(0).to(device)

    # Zero mask = nothing masked. Model only sees its real inputs.
    no_mask = torch.zeros(1, 1, model_cfg.img_size, model_cfg.img_size, device=device)

    recon_seen   = model(x_fixed[0:1], no_mask)
    recon_unseen = model(unseen,       no_mask)

    diff_recon = (recon_seen - recon_unseen).abs().mean().item()
    diff_orig  = (x_fixed[0:1] - unseen).abs().mean().item()

    print(f"Mean abs diff, seen vs unseen RECON:  {diff_recon:.6f}")
    print(f"Mean abs diff, seen vs unseen ORIG:   {diff_orig:.6f}  (reference)")
    print(f"Ratio recon/orig: {diff_recon / (diff_orig + 1e-8):.3f}")
    print("Healthy: recon diff comparable to orig diff.")
    print("Concerning: recon diff << orig diff (output barely depends on input).")

In [ ]:
# Cell C — per-pixel L1 error heatmap for the 4 overfit patches.
# Shows where the reconstruction is wrong, both inside the masked regions
# (loss-bearing) and outside (informational only — these pixels do not
# contribute to the training loss). The mask boundary is drawn in cyan so
# you can tell which is which at a glance.
model.eval()
with torch.no_grad():
    recon = model(x_fixed, fixed_mask)

# Per-pixel L1 averaged over channels -> (B, H, W). Mean (not sum) keeps
# units on the same scale as the printed loss values.
err = (recon - x_fixed).abs().mean(dim=1).cpu().numpy()  # (B, H, W)

# Shared color scale across all 4 patches so the heatmaps are directly
# comparable. Per-patch normalization would hide the fact that one patch
# might be much harder than the others.
vmax = float(err.max())

n_patches = x_fixed.shape[0]
fig, axes = plt.subplots(3, n_patches, figsize=(4 * n_patches, 12))

for i in range(n_patches):
    orig = x_fixed[i].cpu().permute(1, 2, 0).numpy()
    rec  = recon[i].clamp(0.0, 1.0).cpu().permute(1, 2, 0).numpy()  # display clip only
    e    = err[i]                                                   # (H, W)
    m    = fixed_mask[i, 0].cpu().numpy()                           # (H, W), 1 = masked

    in_mask_l1     = (e * m).sum() / (m.sum() + 1e-8)
    out_mask_l1    = (e * (1 - m)).sum() / ((1 - m).sum() + 1e-8)

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f"patch {i}: original")
    axes[0, i].axis("off")

    axes[1, i].imshow(rec)
    axes[1, i].set_title("reconstruction (display-clipped)")
    axes[1, i].axis("off")

    # Heatmap. inferno is sequential and reads "darker = lower" naturally;
    # vmin/vmax pinned so colors mean the same thing across the 4 panels.
    im = axes[2, i].imshow(e, cmap="inferno", vmin=0.0, vmax=vmax)
    # Mask boundary contour. levels=[0.5] picks up the 0/1 boundary.
    axes[2, i].contour(m, levels=[0.5], colors="cyan", linewidths=1.0)
    axes[2, i].set_title(f"|err|  in-mask={in_mask_l1:.4f}  out={out_mask_l1:.4f}")
    axes[2, i].axis("off")
    fig.colorbar(im, ax=axes[2, i], fraction=0.046, pad=0.04)

plt.suptitle("Per-pixel L1 error (cyan = mask boundary; loss is on inside only)",
             y=1.00)
plt.tight_layout()
plt.show()

print(f"Global error stats: max={err.max():.4f}, mean={err.mean():.4f}, "
      f"99th pct={np.percentile(err, 99):.4f}")

In [ ]:
# Cell — Heatmap diagnostic on UNSEEN patches.
# Same layout as the overfit heatmap, but using patches the model never saw
# during the overfit run. This is the most informative single check after an
# overfit: a model that memorized the 4 training patches will reconstruct
# them well but fail catastrophically here, while a model that learned a
# general inpainting prior will degrade gracefully.
#
# Reuse the same args.mask_block_size and args.mask_ratio so numbers are
# directly comparable to the overfit heatmap. New random mask, because there
# is no "fixed mask" associated with patches the model never saw.

# Pick patches the model has NOT seen. The fixed overfit set was indices
# [50, 3, 4, 1000]; pick 4 indices well away from those.
unseen_indices = [200, 500, 1500, 2000]
unseen_indices = [i for i in unseen_indices if i < len(dataset)]  # safety
assert len(unseen_indices) >= 1, "dataset too small for unseen indices"

x_unseen = torch.stack([dataset[i] for i in unseen_indices]).to(device)

# Fresh random mask at the same block size / ratio as training.
# Seed it so this cell is reproducible; rerun without the seed to see
# variance across mask realizations.
torch.manual_seed(cfg.seed + 1)
mask_unseen = random_block_mask(x_unseen, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio).to(device)

model.eval()
with torch.no_grad():
    recon_unseen = model(x_unseen, mask_unseen)
    C = x_unseen.shape[1]

    # Per-patch masked L1, same definition as simmim_l1_loss but reported
    # per sample so you can see which patches the model handles well.
    per_patch_l1 = []
    for i in range(len(x_unseen)):
        m = mask_unseen[i:i+1]
        l1 = ((recon_unseen[i:i+1] - x_unseen[i:i+1]).abs() * m).sum() / (m.sum() * C + 1e-8)
        per_patch_l1.append(l1.item())

# Per-pixel L1 averaged over channels for the heatmap, same as overfit cell.
err = (recon_unseen - x_unseen).abs().mean(dim=1).cpu().numpy()  # (B, H, W)

# Shared color scale across the 4 unseen patches so panels are comparable.
# We DO NOT share scale with the overfit heatmap because absolute error
# levels are expected to differ; what matters here is the spatial pattern.
vmax = float(err.max())

n_patches = x_unseen.shape[0]
fig, axes = plt.subplots(3, n_patches, figsize=(4 * n_patches, 12))

for i in range(n_patches):
    orig = x_unseen[i].cpu().permute(1, 2, 0).numpy()
    rec  = recon_unseen[i].clamp(0.0, 1.0).cpu().permute(1, 2, 0).numpy()  # display only
    e    = err[i]
    m    = mask_unseen[i, 0].cpu().numpy()

    in_mask_l1  = (e * m).sum() / (m.sum() + 1e-8)
    out_mask_l1 = (e * (1 - m)).sum() / ((1 - m).sum() + 1e-8)

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f"unseen idx={unseen_indices[i]}")
    axes[0, i].axis("off")

    axes[1, i].imshow(rec)
    axes[1, i].set_title("reconstruction (display-clipped)")
    axes[1, i].axis("off")

    im = axes[2, i].imshow(e, cmap="inferno", vmin=0.0, vmax=vmax)
    axes[2, i].contour(m, levels=[0.5], colors="cyan", linewidths=1.0)
    axes[2, i].set_title(f"|err|  in={in_mask_l1:.4f}  out={out_mask_l1:.4f}")
    axes[2, i].axis("off")
    fig.colorbar(im, ax=axes[2, i], fraction=0.046, pad=0.04)

plt.suptitle("Per-pixel L1 on UNSEEN patches (cyan = mask boundary)", y=1.00)
plt.tight_layout()
plt.show()

print("\n=== Per-patch masked L1 on unseen patches ===")
for idx, l1 in zip(unseen_indices, per_patch_l1):
    print(f"  idx {idx:5d}: {l1:.6f}")
print(f"  mean: {sum(per_patch_l1) / len(per_patch_l1):.6f}")
print(f"  (compare to fixed-mask training final: 0.006638)")
print(f"\nGlobal error stats: max={err.max():.4f}, mean={err.mean():.4f}, "
      f"99th pct={np.percentile(err, 99):.4f}")

In [ ]:
# Quick sanity check: does unseen L1 correlate with patch foreground mass?
# If yes, the model is failing on the foreground-heavy patches, which is
# the exact failure mode that motivates foreground-biased masking later.
print("idx | mean intensity | masked L1")
for idx, l1 in zip(unseen_indices, per_patch_l1):
    mean_int = dataset[idx].mean().item()
    print(f"  {idx:5d} | {mean_int:.4f}        | {l1:.4f}")

In [ ]:
# Trivial baselines for reference. Computed only on masked positions, same
# normalization as simmim_l1_loss, to be directly comparable to training loss.
model.eval()
with torch.no_grad():
    C = x_fixed.shape[1]

    # Baseline 1: predict zero everywhere.
    pred_zero = torch.zeros_like(x_fixed)
    l1_zero = ((pred_zero - x_fixed).abs() * fixed_mask).sum() / (fixed_mask.sum() * C + 1e-8)

    # Baseline 2: predict the per-image mean of the VISIBLE region.
    # i.e. "I cannot see the masked area, I guess it looks like the rest of the image".
    visible_mean = ((x_fixed * (1 - fixed_mask)).sum(dim=(2, 3), keepdim=True)
                    / ((1 - fixed_mask).sum(dim=(2, 3), keepdim=True) + 1e-8))
    pred_mean = visible_mean.expand_as(x_fixed)
    l1_mean = ((pred_mean - x_fixed).abs() * fixed_mask).sum() / (fixed_mask.sum() * C + 1e-8)

    # Baseline 3: the model.
    recon = model(x_fixed, fixed_mask)
    l1_model = ((recon - x_fixed).abs() * fixed_mask).sum() / (fixed_mask.sum() * C + 1e-8)

    print(f"baseline 'predict zero':           {l1_zero.item():.6f}")
    print(f"baseline 'predict visible mean':   {l1_mean.item():.6f}")
    print(f"model:                             {l1_model.item():.6f}")
    print(f"model improvement over zero-pred:  "
          f"{(1 - l1_model.item() / l1_zero.item()) * 100:.1f}%")

## 6. Next: Full Training

Re-initialize model and train on the full dataset (once overfit check passes).

In [ ]:
train_cfg = SimpleNamespace(
    epochs=200,
    lr=1.5e-4,
    weight_decay=0.05,                # SimMIM default (Xie et al. 2022)
    warmup_epochs=10,
    grad_clip_norm=1.0,               # SimMIM uses 1.0 (Xie et al. 2022)
    # Output
    experiment_name="pretrain_simmim_swin",
    encoder_save_name="pretrained_encoder_simmim.pt",
)


### LR scheduler: linear warmup + cosine decay

In [ ]:
import math

def make_lr_lambda(warmup_epochs, total_epochs):
    """Linear warmup for warmup_epochs, then cosine decay to 0."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_lambda


### Re-initialize model, optimizer, scheduler

In [ ]:
from datetime import datetime
import time

# Timestamped output directory
run_timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
save_dir = os.path.join(
    cfg.output_root,
    f"{train_cfg.experiment_name}_{run_timestamp}"
)
os.makedirs(save_dir, exist_ok=True)
print(f"Output directory: {save_dir}")

# Output directory: ./outputs/pretrain_simmim_swin_2026_04_29_172121

In [ ]:
model = SimMIMSwin(model_cfg).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay,
    betas=(0.9, 0.999),
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    make_lr_lambda(train_cfg.warmup_epochs, train_cfg.epochs),
)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

In [ ]:
n_total   = sum(p.numel() for p in model.parameters())
n_encoder = sum(p.numel() for p in model.swinViT.parameters())
print(f"Model re-initialized.")
print(f"Total params:   {n_total:,}")
print(f"Encoder params: {n_encoder:,}")
print(f"Decoder params: {n_total - n_encoder:,}")

### Experiment metadata + CSV logger

In [ ]:
import csv

experiment_meta = {
    "experiment_name":   train_cfg.experiment_name,
    "start_time":        datetime.now().isoformat(),
    "device":            str(device),
    "gpu_name":          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "pytorch_version":   torch.__version__,

    "n_train":           len(train_loader.dataset),
    "n_val":             len(val_loader.dataset),
    "n_train_batches":   len(train_loader),
    "n_val_batches":     len(val_loader),
    "img_size":          model_cfg.img_size,
    "in_channels":       model_cfg.in_channels,

    "encoder":           "SwinTransformer (MONAI, spatial_dims=2)",
    "feature_size":      model_cfg.feature_size,
    "decoder":           "1x1 Conv + PixelShuffle (SimMIM linear head)",
    "total_params":      n_total,
    "encoder_params":    n_encoder,
    "decoder_params":    n_total - n_encoder,

    "mask_block_size":   overfit_cfg.mask_block_size,
    "mask_ratio":        overfit_cfg.mask_ratio,
    "loss_fn":           "simmim_l1_loss (masked-only L1)",

    "optimizer":         "AdamW",
    "lr":                train_cfg.lr,
    "weight_decay":      train_cfg.weight_decay,
    "batch_size":        data_cfg.batch_size,
    "epochs":            train_cfg.epochs,
    "warmup_epochs":     train_cfg.warmup_epochs,
    "scheduler":         "linear warmup + cosine decay",
    "mixed_precision":   device.type == "cuda",
    "grad_clip_norm":    train_cfg.grad_clip_norm,
    "seed":              cfg.seed,
}

print("=" * 60)
for k, v in experiment_meta.items():
    print(f"  {k:25s}: {v}")
print("=" * 60)

csv_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_log.csv")
csv_fields = [
    "epoch", "train_loss", "val_loss", "lr",
    "epoch_time_s", "train_time_s", "val_time_s",
    "best_val_loss", "best_epoch",
    "grad_norm_mean", "grad_norm_max",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()
print(f"Logging to: {csv_path}")


#### (Optional) Resume from checkpoint

Skip this cell for a fresh run. Set `resume_path` to load a previous
checkpoint and continue training from where it left off.

In [ ]:
import csv as _csv

# Set to a checkpoint path to resume, or None for fresh training.
# resume_path = "./outputs/pretrain_simmim_swin_2026_04_29_172121/best_model.pt"  
# e.g. "./outputs/pretrain_simmim_swin_2026_.../best_model.pt"
resume_path = None

start_epoch = 1
_resumed_history = None

if resume_path is not None:
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["val_loss"]
    best_epoch = ckpt["epoch"]
    print(f"Resumed from epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.6f}")
    print(f"Continuing from epoch {start_epoch}")

    # Load training history from the CSV log in the same directory
    ckpt_dir = os.path.dirname(resume_path)
    csv_candidates = [f for f in os.listdir(ckpt_dir) if f.endswith("_log.csv")]
    if csv_candidates:
        csv_resume_path = os.path.join(ckpt_dir, csv_candidates[0])
        with open(csv_resume_path, "r") as f:
            reader = _csv.DictReader(f)
            _resumed_history = list(reader)
        # Only keep rows up to the checkpoint epoch
        _resumed_history = [r for r in _resumed_history if int(r["epoch"]) < start_epoch]
        print(f"Loaded {len(_resumed_history)} epochs of history from {csv_resume_path}")
    else:
        print("No CSV log found in checkpoint directory, history starts empty")
else:
    print("Fresh training run (no checkpoint loaded)")


### Training Loop

In [ ]:
if resume_path is None:
    best_val_loss  = float("inf")
    best_epoch     = 0
    start_epoch    = 1

# Pre-fill history from previous run's CSV, or start empty
if _resumed_history:
    train_losses = [float(r["train_loss"]) for r in _resumed_history]
    val_losses   = [float(r["val_loss"])   for r in _resumed_history]
    lr_history   = [float(r["lr"])         for r in _resumed_history]
    grad_norms   = [float(r["grad_norm_mean"]) for r in _resumed_history]
    epoch_times  = [float(r["epoch_time_s"])   for r in _resumed_history]
    print(f"Pre-filled {len(train_losses)} epochs of history")
else:
    train_losses = []
    val_losses   = []
    lr_history   = []
    grad_norms   = []
    epoch_times  = []

total_train_start = time.time()


In [ ]:
with torch.no_grad():
    # Cache the diagnostic batch once, outside the loop, ideally.
    # If you have not, grab the first val batch here.
    x_diag = next(iter(val_loader)).to(device, non_blocking=True)
    feats = model.encode_pooled(x_diag)             # (B, enc_ch)
    
    # 1. Per-dim std across the batch. Reports anti-collapse health.
    feat_std = feats.std(dim=0)                     # (enc_ch,)
    n_dead   = (feat_std < 1e-3).sum().item()       # dimensions with no variation
    
    # 2. Effective rank via participation ratio of singular values.
    # Reference: Jing et al., "Understanding Dimensional Collapse in
    # Contrastive Self-supervised Learning", ICLR 2022, arXiv 2110.09348.
    # PR = (sum s_i)^2 / sum(s_i^2). Range [1, D]; higher is better.
    # Center features first because rank of mean-shifted matrix is what
    # matters for clustering distance metrics.
    f_centered = feats - feats.mean(dim=0, keepdim=True)
    s = torch.linalg.svdvals(f_centered.float())    # float32 for stability
    eff_rank = (s.sum() ** 2 / (s ** 2).sum()).item()
    
print(f"  diag | feat_std (min/med/max): "
  f"{feat_std.min():.4f} / {feat_std.median():.4f} / {feat_std.max():.4f} "
  f"| dead_dims: {n_dead}/{feats.shape[1]} "
  f"| effective_rank: {eff_rank:.1f}")


In [ ]:
for epoch in range(start_epoch, train_cfg.epochs + 1):
    # --- Training ---
    model.train()
    running_loss = 0.0
    epoch_grad_norms = []
    train_start = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{train_cfg.epochs} [train]", leave=False)
    for batch_idx, x in enumerate(pbar, 1):
        x = x.to(device, non_blocking=True)
        mask = random_block_mask(x, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            recon = model(x, mask)
            loss = simmim_l1_loss(recon, x, mask)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        total_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=train_cfg.grad_clip_norm
        )
        epoch_grad_norms.append(total_norm.item())
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{running_loss / batch_idx:.6f}")

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)
    train_time = time.time() - train_start
    scheduler.step()

    # --- Validation ---
    model.eval()
    val_running_loss = 0.0
    val_start = time.time()
    with torch.no_grad():
        for x in tqdm(val_loader, desc=f"Epoch {epoch}/{train_cfg.epochs} [val]", leave=False):
            x = x.to(device, non_blocking=True)
            mask = random_block_mask(x, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio)
            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                recon = model(x, mask)
                loss = simmim_l1_loss(recon, x, mask)
            val_running_loss += loss.item()
    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)
    val_time = time.time() - val_start
    # Tier-1 SSL collapse checks. Cheap, run every K epochs on a fixed val batch.
    # We use a fixed batch so numbers are comparable across epochs.
    if epoch == 1 or epoch % 10 == 0:
        with torch.no_grad():
            # Cache the diagnostic batch once, outside the loop, ideally.
            # If you have not, grab the first val batch here.
            x_diag = next(iter(val_loader)).to(device, non_blocking=True)
            feats = model.encode_pooled(x_diag)             # (B, enc_ch)
    
            # 1. Per-dim std across the batch. Reports anti-collapse health.
            feat_std = feats.std(dim=0)                     # (enc_ch,)
            n_dead   = (feat_std < 1e-3).sum().item()       # dimensions with no variation
    
            # 2. Effective rank via participation ratio of singular values.
            # Reference: Jing et al., "Understanding Dimensional Collapse in
            # Contrastive Self-supervised Learning", ICLR 2022, arXiv 2110.09348.
            # PR = (sum s_i)^2 / sum(s_i^2). Range [1, D]; higher is better.
            # Center features first because rank of mean-shifted matrix is what
            # matters for clustering distance metrics.
            f_centered = feats - feats.mean(dim=0, keepdim=True)
            s = torch.linalg.svdvals(f_centered.float())    # float32 for stability
            eff_rank = (s.sum() ** 2 / (s ** 2).sum()).item()
    
        print(f"  diag | feat_std (min/med/max): "
              f"{feat_std.min():.4f} / {feat_std.median():.4f} / {feat_std.max():.4f} "
              f"| dead_dims: {n_dead}/{feats.shape[1]} "
              f"| effective_rank: {eff_rank:.1f}")
    
    # --- Logging ---
    current_lr = scheduler.get_last_lr()[0]
    lr_history.append(current_lr)
    epoch_time = train_time + val_time
    epoch_times.append(epoch_time)
    mean_grad = sum(epoch_grad_norms) / len(epoch_grad_norms)
    max_grad  = max(epoch_grad_norms)
    grad_norms.append(mean_grad)

    improved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        improved = " *best*"
        torch.save({
            "epoch":               epoch,
            "model_state_dict":    model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict":   scaler.state_dict(),
            "val_loss":            val_loss,
            "train_loss":          train_loss,
            "model_cfg":           vars(model_cfg),
            "train_cfg":           {k: v for k, v in vars(train_cfg).items()
                                    if isinstance(v, (int, float, str, bool))},
        }, os.path.join(save_dir, "best_model.pt"))

    print(f"Epoch {epoch:3d}/{train_cfg.epochs} | "
          f"Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
          f"LR: {current_lr:.2e} | "
          f"Time: {epoch_time:.1f}s | "
          f"GradNorm: {mean_grad:.3f}{improved}")


    # ── Periodic collapse check (every 10 epochs) ──
    # Effective rank is the only reliable collapse metric for reconstruction SSL.
    if epoch % 10 == 0 or epoch == start_epoch:
        model.eval()
        _diag_feats = []
        with torch.no_grad():
            for _dx in list(val_loader)[:8]:  # 8 batches, ~256 samples
                _dx = _dx.to(device, non_blocking=True)
                _diag_feats.append(model.encode_pooled(_dx).cpu())
        _Z = torch.cat(_diag_feats, dim=0)
        _Z = _Z - _Z.mean(dim=0, keepdim=True)
        _, _S, _ = torch.svd(_Z)
        _s2 = (_S ** 2) / (_S ** 2).sum()
        _eff_rank = torch.exp(-(_s2 * torch.log(_s2 + 1e-12)).sum()).item()
        print(f"  [collapse check] effective_rank: {_eff_rank:.1f} / {_Z.shape[1]}"
              f"  ({_eff_rank / _Z.shape[1] * 100:.1f}%)"
              f"{'  !!! COLLAPSE' if _eff_rank < _Z.shape[1] * 0.1 else '  OK'}")
        model.train()

    csv_writer.writerow({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": current_lr,
        "epoch_time_s": epoch_time,
        "train_time_s": train_time,
        "val_time_s": val_time,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "grad_norm_mean": mean_grad,
        "grad_norm_max": max_grad,
    })
    csv_file.flush()

total_train_time = time.time() - total_train_start
csv_file.close()
print(f"\nTotal training time: {total_train_time / 60:.1f} min")

### Diagnostic: Reconstruction quality on validation patches

Show 4 unseen patches with their masked input and reconstruction.
If masked regions are poorly filled (especially where puncta were),
the encoder is not learning useful representations.

In [ ]:
model.eval()
val_iter = iter(val_loader)
x_show = next(val_iter)[:4].to(device)
mask_show = random_block_mask(x_show, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio)

with torch.no_grad():
    recon_show = model(x_show, mask_show)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    orig   = x_show[i].cpu().permute(1, 2, 0).numpy()
    masked = (x_show[i] * (1 - mask_show[i])).cpu().permute(1, 2, 0).numpy()
    rec    = recon_show[i].cpu().permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig);   axes[0, i].set_title(f"Val patch {i}: original");  axes[0, i].axis("off")
    axes[1, i].imshow(masked); axes[1, i].set_title("masked input");              axes[1, i].axis("off")
    axes[2, i].imshow(rec);    axes[2, i].set_title("reconstruction");            axes[2, i].axis("off")
plt.suptitle("Validation reconstructions")
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "val_reconstructions.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(save_dir, "val_reconstructions.png"), dpi=200, bbox_inches="tight")
plt.show()


### Diagnostic: Dimensional collapse (SVD of pooled features)

Encode patches *without* masking and pool over spatial dims to get one
descriptor per patch. If the singular value spectrum drops off sharply,
the encoder is only using a few dimensions (collapse). Effective rank
below ~10% of total dims is a red flag.

In [ ]:
from torch.utils.data import Subset

N_DIAG = min(2048, len(dataset))
diag_loader = DataLoader(
    Subset(dataset, range(N_DIAG)),
    batch_size=data_cfg.batch_size,
    shuffle=False,
    num_workers=0,
)

model.eval()
latents = []
with torch.no_grad():
    for x in tqdm(diag_loader, desc="Encoding for collapse check"):
        x = x.to(device, non_blocking=True)
        z_pool = model.encode_pooled(x)         # (B, enc_ch)
        latents.append(z_pool.cpu())

Z = torch.cat(latents, dim=0)                    # (N, enc_ch)
D_lat = Z.shape[1]
print(f"Pooled latent matrix Z: {Z.shape}  (N={Z.shape[0]}, D={D_lat})")

# Center and compute SVD
Z = Z - Z.mean(dim=0, keepdim=True)
_, S, _ = torch.svd(Z)
S_norm = S / S[0]

# Effective rank (exponential of entropy of squared singular values)
s2 = (S ** 2) / (S ** 2).sum()
log_s2 = torch.log(s2 + 1e-12)
effective_rank = torch.exp(-(s2 * log_s2).sum()).item()

print(f"Top-10 normalised singular values: {[round(v, 4) for v in S_norm[:10].tolist()]}")
print(f"Effective rank: {effective_rank:.1f} / {D_lat}  ({effective_rank / D_lat * 100:.1f}% of max)")

if effective_rank < D_lat * 0.1:
    print("WARNING: LIKELY DIMENSIONAL COLLAPSE -- effective rank < 10% of D")
elif effective_rank < D_lat * 0.3:
    print("CAUTION: Moderate collapse -- effective rank < 30% of D")
else:
    print("OK: No significant collapse detected")


In [ ]:
# Save SV spectrum for later comparison with other pretraining methods
np.save(os.path.join(save_dir, "sv_spectrum_simmim.npy"), S_norm.numpy())

collapse_diag = {
    "model": "SimMIMSwin",
    "checkpoint": "best_model.pt",
    "n_patches_encoded": int(Z.shape[0]),
    "latent_dim_pooled": D_lat,
    "effective_rank_pooled": effective_rank,
    "top10_sv_normalized": [round(v, 6) for v in S_norm[:10].tolist()],
}
with open(os.path.join(save_dir, "collapse_diag_simmim.json"), "w") as f:
    json.dump(collapse_diag, f, indent=2)
print(f"Saved: {os.path.join(save_dir, 'collapse_diag_simmim.json')}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(S_norm.numpy(), linewidth=1.5)
axes[0].set_xlabel("Singular value index")
axes[0].set_ylabel("s_i / s_0")
axes[0].set_title("Singular Value Spectrum (linear)")
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(S_norm.numpy(), linewidth=1.5)
axes[1].set_xlabel("Singular value index")
axes[1].set_ylabel("s_i / s_0 (log)")
axes[1].set_title("Singular Value Spectrum (log)")
axes[1].axhline(y=0.01, color="red", linestyle="--", alpha=0.5, label="1% of max")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"SimMIM collapse diagnostic — pooled effective rank: {effective_rank:.1f} / {D_lat}", fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "sv_spectrum_simmim.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(save_dir, "sv_spectrum_simmim.png"), dpi=200, bbox_inches="tight")
plt.show()


### Eval

In [ ]:
# Cell B — collapse check: seen vs unseen patch
# Uses a ZERO mask so the comparison isolates the encoder->decoder mapping
# from any randomness in masking. With a random mask, a non-zero diff could
# come from the inputs OR from the masks differing, which is not what we
# want to measure.
model.eval()
with torch.no_grad():
    unseen_idx = 100
    unseen = dataset[unseen_idx].unsqueeze(0).to(device)

    # Zero mask = nothing masked. Model only sees its real inputs.
    no_mask = torch.zeros(1, 1, model_cfg.img_size, model_cfg.img_size, device=device)

    recon_seen   = model(x_fixed[0:1], no_mask)
    recon_unseen = model(unseen,       no_mask)

    diff_recon = (recon_seen - recon_unseen).abs().mean().item()
    diff_orig  = (x_fixed[0:1] - unseen).abs().mean().item()

    print(f"Mean abs diff, seen vs unseen RECON:  {diff_recon:.6f}")
    print(f"Mean abs diff, seen vs unseen ORIG:   {diff_orig:.6f}  (reference)")
    print(f"Ratio recon/orig: {diff_recon / (diff_orig + 1e-8):.3f}")
    print("Healthy: recon diff comparable to orig diff.")
    print("Concerning: recon diff << orig diff (output barely depends on input).")

In [ ]:
# Cell — Heatmap diagnostic on UNSEEN patches.
# Same layout as the overfit heatmap, but using patches the model never saw
# during the overfit run. This is the most informative single check after an
# overfit: a model that memorized the 4 training patches will reconstruct
# them well but fail catastrophically here, while a model that learned a
# general inpainting prior will degrade gracefully.
#
# Reuse the same args.mask_block_size and args.mask_ratio so numbers are
# directly comparable to the overfit heatmap. New random mask, because there
# is no "fixed mask" associated with patches the model never saw.

# Pick patches the model has NOT seen. The fixed overfit set was indices
# [50, 3, 4, 1000]; pick 4 indices well away from those.
unseen_indices = [50, 500, 1500, 2000]
unseen_indices = [i for i in unseen_indices if i < len(dataset)]  # safety
assert len(unseen_indices) >= 1, "dataset too small for unseen indices"

x_unseen = torch.stack([dataset[i] for i in unseen_indices]).to(device)

# Fresh random mask at the same block size / ratio as training.
# Seed it so this cell is reproducible; rerun without the seed to see
# variance across mask realizations.
torch.manual_seed(cfg.seed + 1)
mask_unseen = random_block_mask(x_unseen, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio).to(device)

model.eval()
with torch.no_grad():
    recon_unseen = model(x_unseen, mask_unseen)
    C = x_unseen.shape[1]

    # Per-patch masked L1, same definition as simmim_l1_loss but reported
    # per sample so you can see which patches the model handles well.
    per_patch_l1 = []
    for i in range(len(x_unseen)):
        m = mask_unseen[i:i+1]
        l1 = ((recon_unseen[i:i+1] - x_unseen[i:i+1]).abs() * m).sum() / (m.sum() * C + 1e-8)
        per_patch_l1.append(l1.item())

# Per-pixel L1 averaged over channels for the heatmap, same as overfit cell.
err = (recon_unseen - x_unseen).abs().mean(dim=1).cpu().numpy()  # (B, H, W)

# Shared color scale across the 4 unseen patches so panels are comparable.
# We DO NOT share scale with the overfit heatmap because absolute error
# levels are expected to differ; what matters here is the spatial pattern.
vmax = float(err.max())

n_patches = x_unseen.shape[0]
fig, axes = plt.subplots(3, n_patches, figsize=(4 * n_patches, 12))

for i in range(n_patches):
    orig = x_unseen[i].cpu().permute(1, 2, 0).numpy()
    rec  = recon_unseen[i].clamp(0.0, 1.0).cpu().permute(1, 2, 0).numpy()  # display only
    e    = err[i]
    m    = mask_unseen[i, 0].cpu().numpy()

    in_mask_l1  = (e * m).sum() / (m.sum() + 1e-8)
    out_mask_l1 = (e * (1 - m)).sum() / ((1 - m).sum() + 1e-8)

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f"unseen idx={unseen_indices[i]}")
    axes[0, i].axis("off")

    axes[1, i].imshow(rec)
    axes[1, i].set_title("reconstruction (display-clipped)")
    axes[1, i].axis("off")

    im = axes[2, i].imshow(e, cmap="inferno", vmin=0.0, vmax=vmax)
    axes[2, i].contour(m, levels=[0.5], colors="cyan", linewidths=1.0)
    axes[2, i].set_title(f"|err|  in={in_mask_l1:.4f}  out={out_mask_l1:.4f}")
    axes[2, i].axis("off")
    fig.colorbar(im, ax=axes[2, i], fraction=0.046, pad=0.04)

plt.suptitle("Per-pixel L1 on UNSEEN patches (cyan = mask boundary)", y=1.00)
plt.tight_layout()
plt.show()

print("\n=== Per-patch masked L1 on unseen patches ===")
for idx, l1 in zip(unseen_indices, per_patch_l1):
    print(f"  idx {idx:5d}: {l1:.6f}")
print(f"  mean: {sum(per_patch_l1) / len(per_patch_l1):.6f}")
print(f"  (compare to fixed-mask training final: 0.006638)")
print(f"\nGlobal error stats: max={err.max():.4f}, mean={err.mean():.4f}, "
      f"99th pct={np.percentile(err, 99):.4f}")

In [ ]:
# Quick sanity check: does unseen L1 correlate with patch foreground mass?
# If yes, the model is failing on the foreground-heavy patches, which is
# the exact failure mode that motivates foreground-biased masking later.
print("idx | mean intensity | masked L1")
for idx, l1 in zip(unseen_indices, per_patch_l1):
    mean_int = dataset[idx].mean().item()
    print(f"  {idx:5d} | {mean_int:.4f}        | {l1:.4f}")

In [ ]:
# Trivial baselines for reference. Computed only on masked positions, same
# normalization as simmim_l1_loss, to be directly comparable to training loss.
model.eval()
with torch.no_grad():
    C = x_fixed.shape[1]

    # Baseline 1: predict zero everywhere.
    pred_zero = torch.zeros_like(x_fixed)
    l1_zero = ((pred_zero - x_fixed).abs() * fixed_mask).sum() / (fixed_mask.sum() * C + 1e-8)

    # Baseline 2: predict the per-image mean of the VISIBLE region.
    # i.e. "I cannot see the masked area, I guess it looks like the rest of the image".
    visible_mean = ((x_fixed * (1 - fixed_mask)).sum(dim=(2, 3), keepdim=True)
                    / ((1 - fixed_mask).sum(dim=(2, 3), keepdim=True) + 1e-8))
    pred_mean = visible_mean.expand_as(x_fixed)
    l1_mean = ((pred_mean - x_fixed).abs() * fixed_mask).sum() / (fixed_mask.sum() * C + 1e-8)

    # Baseline 3: the model.
    recon = model(x_fixed, fixed_mask)
    l1_model = ((recon - x_fixed).abs() * fixed_mask).sum() / (fixed_mask.sum() * C + 1e-8)

    print(f"baseline 'predict zero':           {l1_zero.item():.6f}")
    print(f"baseline 'predict visible mean':   {l1_mean.item():.6f}")
    print(f"model:                             {l1_model.item():.6f}")
    print(f"model improvement over zero-pred:  "
          f"{(1 - l1_model.item() / l1_zero.item()) * 100:.1f}%")

### Save experiment summary

In [ ]:
experiment_meta["end_time"]         = datetime.now().isoformat()
experiment_meta["total_time_hours"] = round(total_train_time / 3600, 3)
experiment_meta["best_val_loss"]    = best_val_loss
experiment_meta["best_epoch"]       = best_epoch
experiment_meta["final_train_loss"] = train_losses[-1]
experiment_meta["final_val_loss"]   = val_losses[-1]

meta_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_meta.json")
with open(meta_path, "w") as f:
    json.dump(experiment_meta, f, indent=2)
print(f"Saved metadata: {meta_path}")


### Loss curves

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_range, train_losses, label="Train")
axes[0].plot(epochs_range, val_losses,   label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("L1 loss (masked)")
axes[0].set_title("Loss (linear)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(epochs_range, train_losses, label="Train")
axes[1].semilogy(epochs_range, val_losses,   label="Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("L1 loss (log)")
axes[1].set_title("Loss (log)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"SimMIM training: {train_cfg.experiment_name}")
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "loss_curves.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(save_dir, "loss_curves.png"), dpi=200, bbox_inches="tight")
plt.show()


### Save encoder weights

Load the best checkpoint and extract just the SwinViT encoder state dict.
This is what gets loaded into `SwinUNETR.swinViT` for fine-tuning.

In [ ]:
best_ckpt = torch.load(
    os.path.join(save_dir, "best_model.pt"),
    map_location=device,
)
model.load_state_dict(best_ckpt["model_state_dict"])

encoder_path = os.path.join(save_dir, train_cfg.encoder_save_name)
torch.save(model.swinViT.state_dict(), encoder_path)
print(f"Saved encoder: {encoder_path}")
print(f"Best epoch: {best_ckpt['epoch']}, val_loss: {best_ckpt['val_loss']:.6f}")
